In [1]:
from vcm_model import  *

In [2]:
##评估指标说明：
##每个指标对应的list按照如下顺序输入：
##第一个元素为指标类型，type1:原子指标；type2:策略不影响分母；type3:策略影响分母
##第二个元素为指标分子（原子指标即为指标本身）
##第三个元素为分子的协变量，list形式，无协变量则置空
##第四个元素为指标分母
##第五个元素为分母的协变量，list形式，无协变量则置空
eval_dict={
    '冒泡数':['type1','bubble_cnt',[]],
    '发单数':['type1','compete_call_cnt_hxz',[]],
    '完单数':['type1','hxz_finish_cnt',['bubble_cnt','valid_tsh']],
    '应答率':['type3','hxz_answer_cnt',['bubble_cnt','valid_tsh'],'compete_call_cnt_hxz',['bubble_cnt','valid_tsh']],
    '成交率':['type3','hxz_finish_cnt',['bubble_cnt','valid_tsh'],'compete_call_cnt_hxz',['bubble_cnt','valid_tsh']],
    '应答后司机取消率(含改派)':['type3','hxz_grab_after_dri_cancel_cnt',['bubble_cnt','valid_tsh'],'hxz_answer_cnt',['bubble_cnt','valid_tsh']],
    '应答后司机取消率(不含改派)':['type3','hxz_grab_after_dri_cancel_cnt_without_reassign',['bubble_cnt','valid_tsh'],'hxz_answer_cnt',['bubble_cnt','valid_tsh']],
    '应答后乘客取消率':['type3','hxz_grab_after_pas_cancel_cnt',['bubble_cnt','valid_tsh'],'hxz_answer_cnt',['bubble_cnt','valid_tsh']],
    '自营完单数':['type1','hxz_zy_finish_cnt',['bubble_cnt','valid_tsh']],
    '宏鹄完单数':['type1','hxz_hh_finish_cnt',['bubble_cnt','valid_tsh']],
    'API完单数':['type1','hxz_api_finish_cnt',['bubble_cnt','valid_tsh']]
}

In [3]:
##评估指标说明：
##每个指标对应的list按照如下顺序输入：
##第一个元素为指标类型，type1:原子指标；type2:策略不影响分母；type3:策略影响分母
##第二个元素为指标分子（原子指标即为指标本身）
##第三个元素为分子的协变量，list形式，无协变量则置空
##第四个元素为指标分母
##第五个元素为分母的协变量，list形式，无协变量则置空
# eval_dict={
#     'GMV':['type1','gmv',['发单量','TSH'],],
#     '单均GMV':['type2','gmv',['TSH'],'发单量'],
#     '应答后取消率':['type3','应答后取消量',[],'应答量',['发单量','TSH']],
# }

In [4]:
##数据格式说明：
##city_id：城市id，必须项
##tdate：实验日期，必须项
##exp_group：实验分组情况，枚举值treatment_group、control_group，必须项
##minf：时间片标识，必须项
##以上必须项的字段名称和实验分组名称，需与demo数据保持一致
df = pd.read_csv('data1.csv')
def chenge_group_name(x):
    return x + '_group'
df['exp_group'] = df['ab_group'].apply(chenge_group_name)
df.head()

,city_id,tdate,minf,ab_group,hxz_finish_cnt,hxz_answer_cnt,hxz_answer_cnt_without_reassign,hxz_grab_after_dri_cancel_cnt,hxz_grab_after_dri_cancel_cnt_without_reassign,hxz_grab_after_pas_cancel_cnt,hxz_grab_after_pas_cancel_cnt_without_reassign,hxz_gmv_amt,hxz_zy_answer_cnt,hxz_zy_finish_cnt,hxz_hh_answer_cnt,hxz_hh_finish_cnt,hxz_api_finish_cnt,pbd_answer_cnt,pbd_finish_cnt,pbd_zy_answer_cnt,pbd_zy_finish_cnt,pbd_hh_answer_cnt,pbd_hh_finish_cnt,pbd_api_finish_cnt,thlm_answer_cnt,thlm_finish_cnt,thlm_zy_answer_cnt,thlm_zy_finish_cnt,thlm_hh_answer_cnt,thlm_hh_finish_cnt,thlm_api_finish_cnt,juhe_zyhh_finish_cnt,juhe_zy_finish_cnt,juhe_zy_gmv_amt,compete_call_cnt_unity,compete_call_cnt_hxz,compete_call_cnt_pbd,compete_call_cnt_zy,compete_call_cnt_hh,compete_call_cnt_zyhh,compete_call_cnt_sf,compete_call_cnt_hxz_zy,compete_call_cnt_hxz_hh,compete_call_cnt_hxz_sf,compete_call_cnt_pbd_zy,compete_call_cnt_pbd_hh,compete_call_cnt_pbd_sf,bubble_cnt,valid_tsh,exp_group
0,2,2025-03-18,6,treatment,386,508,480,42,11,77,74,11885.06000,192,125,45,34,200,340,246,182,121,6,5,25,1254,911,459,298,96,51,562,634,544,17425.21,8248,560,2253,7258,5017,7263,6744,537,520,543,2087,288,975,876,349.00,treatment_group
1,2,2025-03-18,7,control,934,1411,1343,119,17,349,344,31686.36143,614,338,81,38,518,673,419,426,238,14,6,68,3978,2703,1261,718,251,130,1855,1468,1294,42808.13,25039,1640,4922,22981,17604,23003,21341,1610,1564,1598,4534,861,2227,2362,833.00,control_group
2,2,2025-03-18,9,control,1078,1619,1477,179,28,359,348,28137.12572,844,508,24,9,509,756,462,472,243,1,0,104,4351,2990,1894,1091,33,13,1886,1864,1842,47779.46,27036,1700,5839,22873,16853,22887,22952,1578,1540,1672,5180,808,2500,2646,1372.75,control_group
3,2,2025-03-18,14,treatment,1190,1617,1532,122,30,297,281,30193.77000,669,390,7,2,746,799,538,498,315,2,2,70,1861,1335,701,412,6,3,920,1124,1117,24160.07,19326,1740,6902,17422,10706,17436,15070,1705,1674,1716,6373,636,3222,2770,748.40,treatment_group
4,2,2025-03-18,17,control,1204,1659,1553,126,20,319,304,35228.26000,789,491,22,6,639,593,376,414,246,1,0,64,2735,1995,868,532,21,6,1457,1281,1269,32889.11,18137,1700,6113,16083,9805,16097,13732,1581,1535,1665,5576,363,2344,2887,872.45,control_group


In [5]:
##评估结果将自动输出到该目录下的Result文件夹中
import warnings
warnings.filterwarnings("ignore")
vcm = vcm_result(data=df,evaluate_dict=eval_dict)
vcm.run()

  0%|          | 0/10 [00:00<?, ?it/s]

2



  0%|          | 0/11 [00:00<?, ?it/s]

冒泡数



  9%|▉         | 1/11 [00:00<00:06,  1.51it/s]

发单数


... (232 outputs left out)


 91%|█████████ | 10/11 [00:14<00:01,  1.35s/it]

API完单数



100%|██████████| 10/10 [02:32<00:00, 15.22s/it]


,指标,城市,对照组,实验组,P值,绝对提升,相对提升,绝对提升置信区间下界,绝对提升置信区间上界
0,冒泡数,整体,649835.857143,650721.365321,4.206380e-01,885.508178,0.001363,-1269.668848,3040.685204
1,发单数,整体,470693.714286,471878.201108,1.813156e-01,1184.486822,0.002516,-552.303343,2921.276987
2,完单数,整体,343435.142857,344211.897683,5.200242e-02,776.754826,0.002262,-6.750254,1560.259906
3,应答率,整体,0.949500,0.952908,9.972320e-04,0.003408,0.003589,-0.804489,0.811304
4,成交率,整体,0.729636,0.730677,2.193021e-01,0.001041,0.001427,-0.618444,0.620527
5,应答后司机取消率(含改派),整体,0.086599,0.086793,8.361005e-01,0.000194,0.002236,-0.041807,0.042194
6,应答后司机取消率(不含改派),整体,0.009584,0.009168,1.023666e-04,-0.000416,-0.043419,-0.004852,0.004020
7,应答后乘客取消率,整体,0.140941,0.143398,1.127901e-05,0.002457,0.017433,-0.066938,0.071852
8,自营完单数,整体,87303.142857,89058.052150,4.440892e-16,1754.909293,0.020101,1330.603576,2179.215011
9,宏鹄完单数,整体,179922.000000,178675.307303,4.394202e-05,-1246.692697,-0.006929,-1844.756924,-648.628470
